In [4]:
### This code is for utilizing the data from HIVIS
# HIVIS: https://apps.usgs.gov/hivis
# url generation for API: https://waterservices.usgs.gov/test-tools/
#    -> select instantanious values service

# given a list of sites and a time frame this code will create spreadsheets of the sites with hydrological information
# the spreadsheet should also provide a way to get the images, by providing an image url that can be used from an image scraper code

# ENDGOAL of this code base is to create a dataset for training my models

##########################################################################################################################################
import requests
import pandas as pd

#USGS site codes (get from HIVIS)
site_ids = [
    '02146409',    #lil sugar creek, Charlotte, NC
    '01462000',    #Delaware River at Lambertville NJ
]

startDT = '2025-12-31T00:01'
endDT = '2026-01-31T23:59'

def make_spreadsheet(site_id, startDT, endDT, out_file='usgs_data.xlsx'):
    # get json from USGS API
    base_url = "https://waterservices.usgs.gov/nwis/iv/"
    params = {
        "format": 'json',
        "sites": site_id,
        "startDT": startDT,
        "endDT": endDT,
        "siteStatus": 'all',
    }
    r = requests.get(base_url, params=params)
    r.raise_for_status()
    data = r.json()
    
    time_series = data['value']['timeSeries']
    
    dfs = []
    for series in time_series:
        site_name = series["sourceInfo"]["siteName"]
        site_code = series["sourceInfo"]["siteCode"][0]["value"]
        variable = series["variable"]["variableName"]
        unit = series["variable"]["unit"]["unitCode"]

        values = series["values"][0]["value"]
        print(values[:5])
        print(type(values))
        if not values:
            print('no values found')
            continue
        # Build dataframe for this series
        df = pd.DataFrame(values)
        df["dateTime"] = pd.to_datetime(df["dateTime"]).dt.tz_localize(None)  # remove timezone
        df["value"] = pd.to_numeric(df["value"], errors="coerce")
        df["siteName"] = site_name
        df["siteCode"] = site_code
        df["variable"] = variable
        df["unit"] = unit


        dfs.append(df)

    # Combine into one dataframe
    full_df = pd.concat(dfs, ignore_index=True)

    # Save to Excel
    full_df.to_excel(out_file, index=False)

for site_id in site_ids:
    make_spreadsheet(site_id, startDT, endDT, out_file='usgs_data_'+site_id+'.xlsx')
    
    
    

[{'value': '3.75', 'qualifiers': ['P'], 'dateTime': '2025-12-31T00:05:00.000-05:00'}, {'value': '3.75', 'qualifiers': ['P'], 'dateTime': '2025-12-31T00:10:00.000-05:00'}, {'value': '3.75', 'qualifiers': ['P'], 'dateTime': '2025-12-31T00:15:00.000-05:00'}, {'value': '3.75', 'qualifiers': ['P'], 'dateTime': '2025-12-31T00:20:00.000-05:00'}, {'value': '3.75', 'qualifiers': ['P'], 'dateTime': '2025-12-31T00:25:00.000-05:00'}]
<class 'list'>
[{'value': '1.97', 'qualifiers': ['P'], 'dateTime': '2025-12-31T00:05:00.000-05:00'}, {'value': '1.97', 'qualifiers': ['P'], 'dateTime': '2025-12-31T00:10:00.000-05:00'}, {'value': '1.97', 'qualifiers': ['P'], 'dateTime': '2025-12-31T00:15:00.000-05:00'}, {'value': '1.97', 'qualifiers': ['P'], 'dateTime': '2025-12-31T00:20:00.000-05:00'}, {'value': '1.97', 'qualifiers': ['P'], 'dateTime': '2025-12-31T00:25:00.000-05:00'}]
<class 'list'>
[{'value': '614.79', 'qualifiers': ['P'], 'dateTime': '2025-12-31T00:05:00.000-05:00'}, {'value': '614.79', 'qualifier